**Note**: 

> This exercise has been written out in something called a Jupyter Notebook. We'll discuss Jupyter Notebooks in more detail later in this specialization—they are very a powerful tool for data science communication!—but for the time being, the notebook is just a convenient way for us to write out the exercise. You don't need to *do* anything with the notebook except read its contents—just use write your Python code in a regular `.py` file.

# Merging Data to Understand the Relationship between Drug Legalization and Violent Crime

In recent years, many US states have decided to legalize the use of marijuana. 

When these ideas were first proposed, there were many theories about the relationship between crime and the "War on Drugs" (the term given to US efforts to arrest drug users and dealers over the past several decades). 

In this exercise, we're going to test a few of those theories using drug arrest data from the state of California.

**Note: this is one of the most ambitious exercises in this Coursera course! It is quite involved, and requires you to think not just about how to get your pandas code to work, but also to think about where the data comes from what and what you are trying to accomplish!**

Though California has passed a number of laws lessening penalties for marijuana possession over the years, arguably the biggest changes were in  2010, when the state changed the penalty for possessing a small amount of marijuana from a criminal crime to a "civil" penality (meaning those found guilty only had to pay a fine, not go to jail), though possessing, selling, or producing larger quantities remained illegal. Then in 2016, the state fully legalized marijuana for recreational use, not only making possession of small amounts legal, but also creating a regulatory system for producing marijuana for sale. 

Proponents of drug legalization have long argued that the war on drugs contributes to violent crime by creating an opportunity for drug dealers and organized crime to sell and distribute drugs, a business which tends to generate violence when gangs battle over territory. According to this theory, with drug legalization, we should see violent crime decrease after legalization in places where drug arrests had previously been common. In this exercise, we will explore this argument and explore the relationship between drug legalization and violent crime.

**To be clear,** drug legalization is a complex issue and far more study than what we will do here is required to understand its complexities! This exercise is meant to help you think through how to address data science questions programmatically.

## Loading our data pre-legalization

### Exercise 1 

We will begin by examining [county-level data on arrests from California in 2009](https://github.com/nickeubank/practicaldatascience/tree/master/Example_Data/ca), which is derived from data provided by the Office of the California State Attorney General [here](https://openjustice.doj.ca.gov/data). Load the file `ca_arrests_2009.csv`. 

In [63]:
import pandas as pd
pd.set_option("mode.copy_on_write",True)

In [64]:
ca2009=pd.read_csv("ca_arrests_2009.csv")
ca2009.head()

,Unnamed: 0,COUNTY,VIOLENT,PROPERTY,F_DRUGOFF,F_SEXOFF,F_ALLOTHER,F_TOTAL,M_TOTAL,S_TOTAL
0,1682,Alameda County,4318,4640,5749,260,3502,18469,37247,431
1,1683,Alpine County,8,4,2,1,1,16,83,0
2,1684,Amador County,100,59,101,5,199,464,801,2
3,1685,Butte County,641,602,542,34,429,2248,9026,1
4,1686,Calaveras County,211,83,123,14,70,501,968,3


In [65]:
ca2009.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58 entries, 0 to 57
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Unnamed: 0  58 non-null     int64 
 1   COUNTY      58 non-null     object
 2   VIOLENT     58 non-null     int64 
 3   PROPERTY    58 non-null     int64 
 4   F_DRUGOFF   58 non-null     int64 
 5   F_SEXOFF    58 non-null     int64 
 6   F_ALLOTHER  58 non-null     int64 
 7   F_TOTAL     58 non-null     int64 
 8   M_TOTAL     58 non-null     int64 
 9   S_TOTAL     58 non-null     int64 
dtypes: int64(9), object(1)
memory usage: 4.7+ KB


### Exercise 2 

Use your data exploration skills to get a feel for this data. If you need to, you can find the [original codebook here](https://data-openjustice.doj.ca.gov/sites/default/files/dataset/2019-07/Arrests%20Context_062119.pdf) (This data are similar, but have been collapsed to one observation per county.)

In [66]:
ca2018=pd.read_csv("ca_arrests_2018.csv")
ca2009.head()

,Unnamed: 0,COUNTY,VIOLENT,PROPERTY,F_DRUGOFF,F_SEXOFF,F_ALLOTHER,F_TOTAL,M_TOTAL,S_TOTAL
0,1682,Alameda County,4318,4640,5749,260,3502,18469,37247,431
1,1683,Alpine County,8,4,2,1,1,16,83,0
2,1684,Amador County,100,59,101,5,199,464,801,2
3,1685,Butte County,641,602,542,34,429,2248,9026,1
4,1686,Calaveras County,211,83,123,14,70,501,968,3


In [67]:
ca2018.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58 entries, 0 to 57
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Unnamed: 0  58 non-null     int64 
 1   COUNTY      58 non-null     object
 2   VIOLENT     58 non-null     int64 
 3   PROPERTY    58 non-null     int64 
 4   F_DRUGOFF   58 non-null     int64 
 5   F_SEXOFF    58 non-null     int64 
 6   F_ALLOTHER  58 non-null     int64 
 7   F_TOTAL     58 non-null     int64 
 8   M_TOTAL     58 non-null     int64 
 9   S_TOTAL     58 non-null     int64 
dtypes: int64(9), object(1)
memory usage: 4.7+ KB


### Exercise 3 

Figuring out what county has the most violent arrests isn't very meaningful if we don't normalize for size. A county with 10 people and 10 arrests for violent crimes is obviously worse than a county with 1,000,000 people an 11 arrests for violent crime. 

To address this, also import `nhgis_county_populations.csv`.

In [68]:
pop=pd.read_csv("nhgis_county_populations.csv")
pop.head()

,Unnamed: 0,YEAR,STATE,COUNTY,total_population
0,0,2005-2009,Alabama,Autauga County,49584
1,1,2005-2009,Alabama,Baldwin County,171997
2,2,2005-2009,Alabama,Barbour County,29663
3,3,2005-2009,Alabama,Bibb County,21464
4,4,2005-2009,Alabama,Blount County,56804


In [69]:
pop.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6441 entries, 0 to 6440
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Unnamed: 0        6441 non-null   int64 
 1   YEAR              6441 non-null   object
 2   STATE             6441 non-null   object
 3   COUNTY            6441 non-null   object
 4   total_population  6441 non-null   int64 
dtypes: int64(2), object(3)
memory usage: 251.7+ KB


In [70]:
pop.value_counts('YEAR')

YEAR
2005-2009    3221
2013-2017    3220
Name: count, dtype: int64

In [71]:
pop2009=pop[(pop.STATE=='California')&(pop.YEAR=='2005-2009')]
pop2018=pop[(pop.STATE=='California')&(pop.YEAR=='2013-2017')]
pop2009.info()
pop2018.info()

<class 'pandas.core.frame.DataFrame'>
Index: 58 entries, 186 to 243
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Unnamed: 0        58 non-null     int64 
 1   YEAR              58 non-null     object
 2   STATE             58 non-null     object
 3   COUNTY            58 non-null     object
 4   total_population  58 non-null     int64 
dtypes: int64(2), object(3)
memory usage: 2.7+ KB
<class 'pandas.core.frame.DataFrame'>
Index: 58 entries, 3407 to 3464
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Unnamed: 0        58 non-null     int64 
 1   YEAR              58 non-null     object
 2   STATE             58 non-null     object
 3   COUNTY            58 non-null     object
 4   total_population  58 non-null     int64 
dtypes: int64(2), object(3)
memory usage: 2.7+ KB


### Exercise 4 

Use your data exploration skills to get used to these data and figure out how they relates to your 2009 arrest data. Determine the meaning of the various columns and check the data for completeness

In [72]:
ca2009.head(2)

,Unnamed: 0,COUNTY,VIOLENT,PROPERTY,F_DRUGOFF,F_SEXOFF,F_ALLOTHER,F_TOTAL,M_TOTAL,S_TOTAL
0,1682,Alameda County,4318,4640,5749,260,3502,18469,37247,431
1,1683,Alpine County,8,4,2,1,1,16,83,0


In [73]:
pop2009.head(2)

,Unnamed: 0,YEAR,STATE,COUNTY,total_population
186,186,2005-2009,California,Alameda County,1457095
187,187,2005-2009,California,Alpine County,1153


## Merging our data


### Exercise 5

Once you feel like you have a good sense of the relation between our arrest and population data, merge the two datasets. You may need to filter the data first. Do both datasets cover all states or just some or just one? Which years do you care about in this case?

In [74]:
df2009=pd.merge(ca2009,pop2009,how='outer',on='COUNTY',indicator=True)
df2009.head()

,Unnamed: 0_x,COUNTY,VIOLENT,PROPERTY,F_DRUGOFF,F_SEXOFF,F_ALLOTHER,F_TOTAL,M_TOTAL,S_TOTAL,Unnamed: 0_y,YEAR,STATE,total_population,_merge
0,1682,Alameda County,4318,4640,5749,260,3502,18469,37247,431,186,2005-2009,California,1457095,both
1,1683,Alpine County,8,4,2,1,1,16,83,0,187,2005-2009,California,1153,both
2,1684,Amador County,100,59,101,5,199,464,801,2,188,2005-2009,California,38039,both
3,1685,Butte County,641,602,542,34,429,2248,9026,1,189,2005-2009,California,217917,both
4,1686,Calaveras County,211,83,123,14,70,501,968,3,190,2005-2009,California,46548,both


In [75]:
df2009._merge.value_counts()

_merge
both          58
left_only      0
right_only     0
Name: count, dtype: int64

In [76]:
df2018=pd.merge(ca2018,pop2018,how='outer',on='COUNTY',indicator=True)
df2018.head()

,Unnamed: 0_x,COUNTY,VIOLENT,PROPERTY,F_DRUGOFF,F_SEXOFF,F_ALLOTHER,F_TOTAL,M_TOTAL,S_TOTAL,Unnamed: 0_y,YEAR,STATE,total_population,_merge
0,2204,Alameda County,4132,3051,1062,173,2619,11037,28305,82,186,2013-2017,California,1629615,both
1,2205,Alpine County,5,2,1,0,3,11,41,0,187,2013-2017,California,1203,both
2,2206,Amador County,72,40,31,3,142,288,701,1,188,2013-2017,California,37306,both
3,2207,Butte County,785,437,229,47,741,2239,8853,1,189,2013-2017,California,225207,both
4,2208,Calaveras County,147,42,29,6,96,320,897,0,190,2013-2017,California,45057,both


In [77]:
df2018._merge.value_counts()

_merge
both          58
left_only      0
right_only     0
Name: count, dtype: int64


### Exercise 6 

Now repeat your previous merge using *both* the `validate` keyword *and* the `indicator` keyword with `how="outer"` as discussed in the last reading to help you debug your merge. (Hint: your merge should end up being a one to one merge)

In [78]:
df2009=pd.merge(ca2009,pop2009,how='outer',on='COUNTY',indicator=True,validate='1:1')
df2009.head()
#df2009._merge.value_counts()

,Unnamed: 0_x,COUNTY,VIOLENT,PROPERTY,F_DRUGOFF,F_SEXOFF,F_ALLOTHER,F_TOTAL,M_TOTAL,S_TOTAL,Unnamed: 0_y,YEAR,STATE,total_population,_merge
0,1682,Alameda County,4318,4640,5749,260,3502,18469,37247,431,186,2005-2009,California,1457095,both
1,1683,Alpine County,8,4,2,1,1,16,83,0,187,2005-2009,California,1153,both
2,1684,Amador County,100,59,101,5,199,464,801,2,188,2005-2009,California,38039,both
3,1685,Butte County,641,602,542,34,429,2248,9026,1,189,2005-2009,California,217917,both
4,1686,Calaveras County,211,83,123,14,70,501,968,3,190,2005-2009,California,46548,both


In [79]:
df2018=pd.merge(ca2018,pop2018,how='outer',on='COUNTY',indicator=True,validate='1:1')
df2018.head()
#df2018._merge.value_counts()

,Unnamed: 0_x,COUNTY,VIOLENT,PROPERTY,F_DRUGOFF,F_SEXOFF,F_ALLOTHER,F_TOTAL,M_TOTAL,S_TOTAL,Unnamed: 0_y,YEAR,STATE,total_population,_merge
0,2204,Alameda County,4132,3051,1062,173,2619,11037,28305,82,186,2013-2017,California,1629615,both
1,2205,Alpine County,5,2,1,0,3,11,41,0,187,2013-2017,California,1203,both
2,2206,Amador County,72,40,31,3,142,288,701,1,188,2013-2017,California,37306,both
3,2207,Butte County,785,437,229,47,741,2239,8853,1,189,2013-2017,California,225207,both
4,2208,Calaveras County,147,42,29,6,96,320,897,0,190,2013-2017,California,45057,both


### Exercise 7

You *should* be able to get to the point that all counties in our arrest data merge with population data. Can you figure out why that did not happen? Using the tools we just discussed, look for any inconsistencies across the two datasets to see if anything did not match when it should have. 

**You will need to fix the data so that all 58 counties in the arrest data merge with population data for your subsequent answers to be correct.** 

The type of data edit we're asking you to make may make you feel a little uncomfortable — who are you to edit the data, after all?! The answer is: you're the data scientist who has to make sense of this data! When merging data — especially data that uses names stored as strings — you will often discover different datasets have found slightly different ways to label observations. It's up to you to use your critical thinking skills, context clues, and domain knowledge to evaluate whether you think observations in different datasets are actually the same entity.

*Hint: what are the DataFrames being merged on? Does it match across both DataFrames?*

## Calculating arrest rates and gathering 2018 data

### Exercise 8 

Now that we have arrest counts and population data, we can calculate arrest *rates*. For each county, create a new variable called `violent_arrest_rate_2009` that is the number of violent arrests for 2009 divided by the population of the county from 2005-2009, and an analogous variable for drug offenses (`F_DRUGOFF`) called `f_drugoff_arrest_rate_2009`. 

In general, people tend not to be arrested that often as a share of population, so to avoid working with tiny numbers, statistics like arrest rates are often reported as arrests per X people (rather than arrests per capita, which is just arrests divided by population).

**For this and all following exercises, please calculate arrest rates as arrests per 1,000 people**.

Calculate the average county-level felony drug arrest rate for 2009 (in arrests per 1,000) rounded to three significant places. **Note this average value - you will need to submit this at the end of this week for the final Quiz.**

In [80]:
df2009=df2009.drop(columns=['Unnamed: 0_y','STATE','_merge'],errors='ignore')
df2018=df2018.drop(columns=['Unnamed: 0_y','STATE','_merge'],errors='ignore')

In [85]:
df2009['violent_arrest_rate_2009']=(df2009['VIOLENT']/(df2009['total_population']/1000))
df2009['f_drugoff_arrest_rate_2009']=(df2009['F_DRUGOFF']/(df2009['total_population']/1000))
df2009.head()

,Unnamed: 0_x,COUNTY,VIOLENT,PROPERTY,F_DRUGOFF,F_SEXOFF,F_ALLOTHER,F_TOTAL,M_TOTAL,S_TOTAL,YEAR,total_population,violent_arrest_rate_2009,f_drugoff_arrest_rate_2009
0,1682,Alameda County,4318,4640,5749,260,3502,18469,37247,431,2005-2009,1457095,2.963431,3.945522
1,1683,Alpine County,8,4,2,1,1,16,83,0,2005-2009,1153,6.938422,1.734605
2,1684,Amador County,100,59,101,5,199,464,801,2,2005-2009,38039,2.628881,2.655170
3,1685,Butte County,641,602,542,34,429,2248,9026,1,2005-2009,217917,2.941487,2.487185
4,1686,Calaveras County,211,83,123,14,70,501,968,3,2005-2009,46548,4.532955,2.642434


In [88]:
print("Violent arrest rate:",df2009.violent_arrest_rate_2009.mean())
print("Felony Drug Arrest:",df2009.f_drugoff_arrest_rate_2009.mean())

Violent arrest rate: 3.710096949943832
Felony Drug Arrest: 3.1914480151583384


### Exercise 9 

Just as we created violent arrest rates and drug arrest rates for 2009, now we want to do it for 2018, so we can work towards comparing the two. Using the data on 2018 arrests (ca_arrests_2018.csv) and the same dataset of population data (you'll use population from 2013-2017 this time), create a dataset of arrest rates. 

As before, be careful with your merges! The same issues you uncovered with the 2009 data are likely to also be present here, **and if you don't correct them again, your following answers will be incorrect.** 

If you: 

(a) don't end up with population data for all 58 counties, or 

(b) don't get your merge to be 1-to-1

something is wrong.

### Exercise 10 

Go ahead and calculate the arrest rates for the 2018 dataset as well. For each county, create a new variable called `violent_arrest_rate_2018` that is the number of violent arrests for 2018 divided by the population of the county from 2013-2017, and an analogous variable for drug offenses (`F_DRUGOFF`) called `f_drugoff_arrest_rate_2018`. 

**For this and all following exercises, please calculate arrest rates as arrests per 1,000 people**.

Calculate the average county-level felony drug arrest rate for 2018 (in arrests per 1,000) rounded to three significant places. **Note this average value - you will need to submit this at the end of this week for the final Quiz.**

In [90]:
df2018['violent_arrest_rate_2018']=((df2018['VIOLENT']/df2018['total_population'])*1000)
df2018['f_drugoff_arrest_rate_2018']=((df2018['F_DRUGOFF']/df2018['total_population'])*1000)
print("Felony Drug Arrest:",df2018.f_drugoff_arrest_rate_2018.mean())

Felony Drug Arrest: 0.9781007363217146


## Comparing 2009 with 2018 Arrests: Repeating your merge from the 2009 data

If we plotted our rate data for 2009 (violent crime arrest rate vs felony drug arrest rate) it would show that drug arrests and violent crime arrests tend to be positively correlated, but that does not tell us much about whether they are *causally* related. It *could* be the case that people dealing drugs *causes* more violent crime, but it could also be that certain communities, for some other reason, tend to have *both* more drug sales *and* more violent crime. 

So to test for this, we went to see if the same communities that had violent crime in 2009 *also* have violent crime in 2019 (after marijuana legalization). If these communities have just as much crime in 2018, that would suggest that violent crime is being driven by a third factor, and not drug sales of marijuana. 

### Exercise 11 

Merge the two county-level datasets so you have one row for each county, and variables for violent arrest rates in 2018, violent arrest rates in 2009, felony drug arrest rates in 2018, and felony drug arrest rates in 2009. You will need at least 5 columns from this going forward (you're welcome to drop the rest for the remainder of the analysis):

1. COUNTY
2. violent_arrest_rate_2009
3. violent_arrest_rate_2018
4. f_drug_arrest_rate_2009
4. f_drug_arrest_rate_2018

*Hints and notes*: 

- If you used `indicator = True`, you may need to drop the `_merge` columns before merging from each dataset
- Since you'll be merging two DataFrames with the same column names, when you merge them it will create two versions from each dataset, one from the first DataFrame you list in the merge (which will be appended with '_x' in the column name and one from the second DataFrame you list in the merge which will be appended with '_y' in the column name).
- At any time you can use the `rename` method in pandas to adjust column names if it makes them easier for you to understand

In [91]:
df=pd.merge(df2009,df2018,how='outer',on='COUNTY',indicator=True,validate='1:1')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58 entries, 0 to 57
Data columns (total 28 columns):
 #   Column                      Non-Null Count  Dtype   
---  ------                      --------------  -----   
 0   Unnamed: 0_x_x              58 non-null     int64   
 1   COUNTY                      58 non-null     object  
 2   VIOLENT_x                   58 non-null     int64   
 3   PROPERTY_x                  58 non-null     int64   
 4   F_DRUGOFF_x                 58 non-null     int64   
 5   F_SEXOFF_x                  58 non-null     int64   
 6   F_ALLOTHER_x                58 non-null     int64   
 7   F_TOTAL_x                   58 non-null     int64   
 8   M_TOTAL_x                   58 non-null     int64   
 9   S_TOTAL_x                   58 non-null     int64   
 10  YEAR_x                      58 non-null     object  
 11  total_population_x          58 non-null     int64   
 12  violent_arrest_rate_2009    58 non-null     float64 
 13  f_drugoff_arrest_rate_

In [92]:
df.head(2)

,Unnamed: 0_x_x,COUNTY,VIOLENT_x,PROPERTY_x,F_DRUGOFF_x,F_SEXOFF_x,F_ALLOTHER_x,F_TOTAL_x,M_TOTAL_x,S_TOTAL_x,...,F_SEXOFF_y,F_ALLOTHER_y,F_TOTAL_y,M_TOTAL_y,S_TOTAL_y,YEAR_y,total_population_y,violent_arrest_rate_2018,f_drugoff_arrest_rate_2018,_merge
0,1682,Alameda County,4318,4640,5749,260,3502,18469,37247,431,...,173,2619,11037,28305,82,2013-2017,1629615,2.535568,0.651688,both
1,1683,Alpine County,8,4,2,1,1,16,83,0,...,0,3,11,41,0,2013-2017,1203,4.156276,0.831255,both


In [93]:
df=df[['COUNTY','violent_arrest_rate_2009','violent_arrest_rate_2018','f_drugoff_arrest_rate_2009','f_drugoff_arrest_rate_2018']]
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58 entries, 0 to 57
Data columns (total 5 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   COUNTY                      58 non-null     object 
 1   violent_arrest_rate_2009    58 non-null     float64
 2   violent_arrest_rate_2018    58 non-null     float64
 3   f_drugoff_arrest_rate_2009  58 non-null     float64
 4   f_drugoff_arrest_rate_2018  58 non-null     float64
dtypes: float64(4), object(1)
memory usage: 2.4+ KB


In [94]:
df.head()

,COUNTY,violent_arrest_rate_2009,violent_arrest_rate_2018,f_drugoff_arrest_rate_2009,f_drugoff_arrest_rate_2018
0,Alameda County,2.963431,2.535568,3.945522,0.651688
1,Alpine County,6.938422,4.156276,1.734605,0.831255
2,Amador County,2.628881,1.929984,2.655170,0.830966
3,Butte County,2.941487,3.485682,2.487185,1.016842
4,Calaveras County,4.532955,3.262534,2.642434,0.643629


### Exercise 12 

Did drug arrests go down from 2009 to 2018? (they sure better! This is what's called a "sanity check" of your data and analysis. If you find drug arrests went *up*, you know something went wrong with your code or your understanding of the situations. To verify this, compute the difference between the 2018 drug rate and that of 2009 and review those values sorted from smallest to largest. How many of the values were less than zero (meaning the rate decreased). For how many counties did the rate increase? Calculate the average percentage change in felony drug arrests across all counties.

As a reminder, percentage change can be calculated as follows where $x_{2018}$ is the respective rate for the year 2018:
$$\frac{x_{2018} - x_{2009}}{x_{2009}} \times 100$$

**Note this average percentage change in the felony drug arrest rate value - you will need to submit this at the end of this week for the final Quiz.**

In [108]:
df['violent_change']=(df['violent_arrest_rate_2018']-df['violent_arrest_rate_2009'])
df['drugoff_change']=(df['f_drugoff_arrest_rate_2018']-df['f_drugoff_arrest_rate_2009'])
print(df.head())

             COUNTY  violent_arrest_rate_2009  violent_arrest_rate_2018  \
0    Alameda County                  2.963431                  2.535568   
1     Alpine County                  6.938422                  4.156276   
2     Amador County                  2.628881                  1.929984   
3      Butte County                  2.941487                  3.485682   
4  Calaveras County                  4.532955                  3.262534   

   f_drugoff_arrest_rate_2009  f_drugoff_arrest_rate_2018  violent_change  \
0                    3.945522                    0.651688       -0.427862   
1                    1.734605                    0.831255       -2.782146   
2                    2.655170                    0.830966       -0.698896   
3                    2.487185                    1.016842        0.544195   
4                    2.642434                    0.643629       -1.270421   

   drugoff_change  
0       -3.293834  
1       -0.903350  
2       -1.824204  
3     

In [109]:
(((df['drugoff_change'])/(df['f_drugoff_arrest_rate_2009']))*100).mean()

-66.31166718090417

In [110]:
(((df.f_drugoff_arrest_rate_2018.mean())-(df.f_drugoff_arrest_rate_2009.mean()))/(df.f_drugoff_arrest_rate_2009.mean()))*100

-69.35244654852421

### Exercise 13 

Now we want to look at whether violent crime decreased following drug legalization. Did the average violent arrest rate decrease? By how much? (Note: We're assuming that arrest rates are proportionate to crime rates. If policing increased so that there were more arrests per crime committed, that would impact our interpretation of these results. But this is just an exercise, so we'll keep it simple)

**Note this average percentage change in the violent crime arrest rate value - you will need to submit this at the end of this week for the final Quiz.**

In [111]:
(((df['violent_change'])/(df['violent_arrest_rate_2009']))*100).mean()

-6.77098912074797

## Diving deeper into the post-legalization changes

### Exercise 14 

So we've determined that both drug arrests and violent crime arrests were decreasing over this period. But maybe *all* crime was just falling, and this isn't about drug legalization. 

This is the problem with a "pre-to-post" analysis: yes, our results are *consistent* with the idea that drug legalization reduced violent crime, but lots of things happened between 2009 and 2018, not just drug legalization, so we don't know that drug legalization *caused* the decline in violent crime. 

So let's do a kind of difference-in-difference analysis. We know that drug legalization should have had a bigger effect on counties that had higher drug arrest rates prior to drug legalization. After all, in a county that had no drug arrests, legalization wouldn't do anything, would it? 

So let's split our sample into two groups: high drug arrests in 2009, and low drug arrests in 2009. 

To decide who goes into each group, first calculate the average 2009 drug arrest rate across all counties. Then make the "high drug arrest" group counties whose 2009 drug arrest rate was above that mean value and make the "low drug arrest rate" counties whose 2009 drug arrest rate was above that mean value.

In [112]:
df.head()

,COUNTY,violent_arrest_rate_2009,violent_arrest_rate_2018,f_drugoff_arrest_rate_2009,f_drugoff_arrest_rate_2018,violent_change,drugoff_change
0,Alameda County,2.963431,2.535568,3.945522,0.651688,-0.427862,-3.293834
1,Alpine County,6.938422,4.156276,1.734605,0.831255,-2.782146,-0.903350
2,Amador County,2.628881,1.929984,2.655170,0.830966,-0.698896,-1.824204
3,Butte County,2.941487,3.485682,2.487185,1.016842,0.544195,-1.470343
4,Calaveras County,4.532955,3.262534,2.642434,0.643629,-1.270421,-1.998804


In [116]:
mean_drug_arrest_2009=df.f_drugoff_arrest_rate_2009.mean()
mean_drug_arrest_2009

3.1914480151583384

In [118]:
df['drug_arrest_group_2009']=df['f_drugoff_arrest_rate_2009'].apply(lambda x: "High" if (x > mean_drug_arrest_2009) else "Low"  )
df.head(10)

,COUNTY,violent_arrest_rate_2009,violent_arrest_rate_2018,f_drugoff_arrest_rate_2009,f_drugoff_arrest_rate_2018,violent_change,drugoff_change,drug_arrest_group_2009
0,Alameda County,2.963431,2.535568,3.945522,0.651688,-0.427862,-3.293834,High
1,Alpine County,6.938422,4.156276,1.734605,0.831255,-2.782146,-0.903350,Low
2,Amador County,2.628881,1.929984,2.655170,0.830966,-0.698896,-1.824204,Low
3,Butte County,2.941487,3.485682,2.487185,1.016842,0.544195,-1.470343,Low
4,Calaveras County,4.532955,3.262534,2.642434,0.643629,-1.270421,-1.998804,Low
5,Colusa County,2.761773,3.072769,1.333270,0.232786,0.310996,-1.100484,Low
6,Contra Costa County,2.930371,2.326289,2.850613,0.708388,-0.604082,-2.142225,Low
7,DelNorte County,5.012357,5.502514,2.749835,1.566941,0.490158,-1.182893,Low
8,El Dorado County,3.239722,2.594384,2.290541,0.551307,-0.645338,-1.739234,Low
9,Fresno County,4.913837,4.442084,3.938254,0.571213,-0.471753,-3.367041,High


### Exercise 15 

Now we can ask: did violent crime fall *more* from 2009 to 2018 in the counties that had lots of drug arrests in 2009 (where legalization likely had more of an effect) than in counties with fewer drug arrests in 2009 (where legalization likely mattered less)? Calculate this using what we call a difference-in-differences, which can be computed as follows:

(the change in violent crime rate for counties with lots of drug arrests in 2009) - (the change in violent crime rate for counties with few drug arrests in 2009)

**Please make sure to calculate arrest rates in arrests per 1,000 people.**

**Note this average value - you will need to submit this at the end of this week for the final Quiz.**

**Note your output here: the percentage change for the case of both the high and the low 2009 drug arrest rate groups in 2009 - you will need to submit this at the end of this week for the final Quiz.**

In [124]:
df['violent_rate_change']=df['violent_arrest_rate_2009']-df['violent_arrest_rate_2018']
df.head()

,COUNTY,violent_arrest_rate_2009,violent_arrest_rate_2018,f_drugoff_arrest_rate_2009,f_drugoff_arrest_rate_2018,violent_change,drugoff_change,drug_arrest_group_2009,violent_rate_change
0,Alameda County,2.963431,2.535568,3.945522,0.651688,-0.427862,-3.293834,High,0.427862
1,Alpine County,6.938422,4.156276,1.734605,0.831255,-2.782146,-0.903350,Low,2.782146
2,Amador County,2.628881,1.929984,2.655170,0.830966,-0.698896,-1.824204,Low,0.698896
3,Butte County,2.941487,3.485682,2.487185,1.016842,0.544195,-1.470343,Low,-0.544195
4,Calaveras County,4.532955,3.262534,2.642434,0.643629,-1.270421,-1.998804,Low,1.270421


In [125]:
df.groupby('drug_arrest_group_2009')['violent_rate_change'].mean()

drug_arrest_group_2009
High    0.431974
Low     0.184580
Name: violent_rate_change, dtype: float64

### Execise 16 

Hmmm... we showed that there was a greater *absolute* decline in violent arrest rates in counties more impacted by drug legalization. But was there also a greater *proportionate* decline?

Repeat the above calculation but for percentage change:

(the percentage change in violent crime rate for counties with lots of drug arrests in 2009) - (the percentage change in violent crime rate for counties with few drug arrests in 2009)

**Note your output here: the percentage change for the case of both the high and the low 2009 drug arrest rate groups in 2009 - you will need to submit this at the end of this week for the final Quiz.**

In [128]:
df['violent_change_pct']=(df['violent_rate_change']/df['violent_arrest_rate_2009'])*100
df.head()

,COUNTY,violent_arrest_rate_2009,violent_arrest_rate_2018,f_drugoff_arrest_rate_2009,f_drugoff_arrest_rate_2018,violent_change,drugoff_change,drug_arrest_group_2009,violent_rate_change,violent_change_pct
0,Alameda County,2.963431,2.535568,3.945522,0.651688,-0.427862,-3.293834,High,0.427862,14.438078
1,Alpine County,6.938422,4.156276,1.734605,0.831255,-2.782146,-0.903350,Low,2.782146,40.097672
2,Amador County,2.628881,1.929984,2.655170,0.830966,-0.698896,-1.824204,Low,0.698896,26.585321
3,Butte County,2.941487,3.485682,2.487185,1.016842,0.544195,-1.470343,Low,-0.544195,-18.500683
4,Calaveras County,4.532955,3.262534,2.642434,0.643629,-1.270421,-1.998804,Low,1.270421,28.026333


In [129]:
df.groupby('drug_arrest_group_2009')['violent_change_pct'].mean()

drug_arrest_group_2009
High    9.736210
Low     4.822416
Name: violent_change_pct, dtype: float64

### Exercise 17 

What are your conclusions about the relationship between violent crime and drug legalization, give your analysis above?